In [33]:
import os

The following code creates and connects to a database file.

In [34]:
import duckdb

con = duckdb.connect("../data/mot.duckdb")

Now, send a query to the database.

In [35]:
con.sql("""
    SELECT *
    FROM read_csv('../data/raw/MOT testing data results (2024)/test_result_202401.csv')
    LIMIT 5
""").show()

┌───────────┬────────────┬────────────┬───────────────┬───────────┬─────────────┬──────────────┬───────────────┬─────────┬─────────┬─────────┬───────────┬───────────────────┬────────────────┬──────────────────────────┐
│  test_id  │ vehicle_id │ test_date  │ test_class_id │ test_type │ test_result │ test_mileage │ postcode_area │  make   │  model  │ colour  │ fuel_type │ cylinder_capacity │ first_use_date │      completed_date      │
│   int64   │   int64    │    date    │     int64     │  varchar  │   varchar   │    double    │    varchar    │ varchar │ varchar │ varchar │  varchar  │       int64       │      date      │ timestamp with time zone │
├───────────┼────────────┼────────────┼───────────────┼───────────┼─────────────┼──────────────┼───────────────┼─────────┼─────────┼─────────┼───────────┼───────────────────┼────────────────┼──────────────────────────┤
│ 886090729 │ 1182957512 │ 2024-01-02 │             4 │ NT        │ P           │      71614.0 │ OX            │ PORSCHE │ 9

The following code describes what the variable types are from a batch.

In [36]:
con.sql("""
    DESCRIBE
    SELECT *
    FROM read_csv('../data/raw/MOT testing data results (2024)/*.csv')
""").show()

┌───────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │       column_type        │  null   │   key   │ default │  extra  │
│      varchar      │         varchar          │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ test_id           │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ vehicle_id        │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ test_date         │ DATE                     │ YES     │ NULL    │ NULL    │ NULL    │
│ test_class_id     │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ test_type         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ test_result       │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ test_mileage      │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode_area     │

Store RESULTS to save reference later. Quote to define quotes. Escape tells DuckDB that a backslash marks the next quote as literal text.

In [37]:
RESULTS = r"""read_csv(
    '../data/raw/MOT testing data results (2024)/*.csv',
    all_varchar=true,
    quote='"',
    escape='\',
    filename=true
)"""

The following query finds the amount of header rows (180) and individual tests (42mil). Total rows - distinct tests = amount of copies.

In [38]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE test_id = 'test_id') AS header_rows,
           COUNT(DISTINCT test_id) FILTER (WHERE test_id <> 'test_id') AS distinct_tests
    FROM {RESULTS}
""").show()

┌────────────┬─────────────┬────────────────┐
│ total_rows │ header_rows │ distinct_tests │
│   int64    │    int64    │     int64      │
├────────────┼─────────────┼────────────────┤
│   66857535 │         180 │       42598624 │
└────────────┴─────────────┴────────────────┘



Shows total rows and number of header rows for each file.

In [39]:
con.sql(f"""
    SELECT filename,
           COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE test_id = 'test_id') AS header_rows
    FROM {RESULTS}
    GROUP BY filename
    ORDER BY filename
""").show()

┌────────────────────────────────────────────────────────────────────┬────────────┬─────────────┐
│                              filename                              │ total_rows │ header_rows │
│                              varchar                               │   int64    │    int64    │
├────────────────────────────────────────────────────────────────────┼────────────┼─────────────┤
│ ../data/raw/MOT testing data results (2024)/test_result_202401.csv │    6140803 │          15 │
│ ../data/raw/MOT testing data results (2024)/test_result_202402.csv │    5858363 │          15 │
│ ../data/raw/MOT testing data results (2024)/test_result_202403.csv │    6513962 │          15 │
│ ../data/raw/MOT testing data results (2024)/test_result_202404.csv │    5101251 │          15 │
│ ../data/raw/MOT testing data results (2024)/test_result_202405.csv │    4997498 │          15 │
│ ../data/raw/MOT testing data results (2024)/test_result_202406.csv │    5140337 │          15 │
│ ../data/raw/MOT te

One again saving JAN to reduce long requests.

In [40]:
JAN = r"""read_csv(
    '../data/raw/MOT testing data results (2024)/test_result_202401.csv',
    all_varchar=true,
    quote='"',
    escape='\'
)"""

WITH t AS (...) creates a named temporary result called t (a CTE), here “January’s data without the repeated headers”. It saves repeating the long query. This is the technique I listed for Step 4, so you’re getting an early look at it.

SELECT DISTINCT * removes rows where every column is identical, and the outer COUNT(*) counts what’s left.

Read the three numbers together:
distinct_full_rows equals distinct_tests: every duplicate is an exact copy.
distinct_full_rows is bigger than distinct_tests: some tests appear with differing values.

In [41]:
con.sql(f"""
    WITH t AS (
        SELECT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT COUNT(*) AS data_rows,
           COUNT(DISTINCT test_id) AS distinct_tests,
           (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM t)) AS distinct_full_rows
    FROM t
""").show()

┌───────────┬────────────────┬────────────────────┐
│ data_rows │ distinct_tests │ distinct_full_rows │
│   int64   │     int64      │       int64        │
├───────────┼────────────────┼────────────────────┤
│   6140788 │        3783238 │            3848449 │
└───────────┴────────────────┴────────────────────┘



Count how mnay instances of repeated tests in dataset.

In [42]:
con.sql(f"""
    WITH t AS (
        SELECT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT copies, COUNT(*) AS n_tests
    FROM (
        SELECT test_id, COUNT(*) AS copies
        FROM t
        GROUP BY test_id
    )
    GROUP BY copies
    ORDER BY copies
""").show()

┌────────┬─────────┐
│ copies │ n_tests │
│ int64  │  int64  │
├────────┼─────────┤
│      1 │ 2611399 │
│      2 │  808888 │
│      3 │   73454 │
│      4 │  182819 │
│      5 │    1008 │
│      6 │   45639 │
│      7 │     116 │
│      8 │   26190 │
│      9 │   10976 │
│     10 │    1612 │
│      · │       · │
│      · │       · │
│      · │       · │
│    130 │       1 │
│    144 │       1 │
│    154 │       1 │
│    176 │       1 │
│    221 │       1 │
│    240 │       1 │
│    252 │       1 │
│    308 │       1 │
│    441 │       1 │
│   1517 │       1 │
└────────┴─────────┘
      69 rows     
     (20 shown)    



The inner query goes through each test_id and counts how many different values appear in each column. A column showing 1 distinct value means all copies agree. More than 1 means they conflict.

The outer query then counts how many tests have a conflict in each column, using FILTER like before.

Read across the output: if only completed_date_differs (and maybe mileage_differs) is large, the conflicts are probably re-recorded versions of the same test. If result_differs is large, that’s the more worrying case, since the same test would have both a pass and a fail on record.

In [43]:
con.sql(f"""
    WITH t AS (
        SELECT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT
        COUNT(*) AS tests,
        COUNT(*) FILTER (WHERE n_rows > 1)       AS tests_with_copies,
        COUNT(*) FILTER (WHERE n_vehicle > 1)    AS vehicle_id_differs,
        COUNT(*) FILTER (WHERE n_date > 1)       AS test_date_differs,
        COUNT(*) FILTER (WHERE n_result > 1)     AS result_differs,
        COUNT(*) FILTER (WHERE n_mileage > 1)    AS mileage_differs,
        COUNT(*) FILTER (WHERE n_make > 1)       AS make_differs,
        COUNT(*) FILTER (WHERE n_fuel > 1)       AS fuel_differs,
        COUNT(*) FILTER (WHERE n_completed > 1)  AS completed_date_differs
    FROM (
        SELECT test_id,
               COUNT(*)                       AS n_rows,
               COUNT(DISTINCT vehicle_id)     AS n_vehicle,
               COUNT(DISTINCT test_date)      AS n_date,
               COUNT(DISTINCT test_result)    AS n_result,
               COUNT(DISTINCT test_mileage)   AS n_mileage,
               COUNT(DISTINCT make)           AS n_make,
               COUNT(DISTINCT fuel_type)      AS n_fuel,
               COUNT(DISTINCT completed_date) AS n_completed
        FROM t
        GROUP BY test_id
    )
""").show()

┌─────────┬───────────────────┬────────────────────┬───────────────────┬────────────────┬─────────────────┬──────────────┬──────────────┬────────────────────────┐
│  tests  │ tests_with_copies │ vehicle_id_differs │ test_date_differs │ result_differs │ mileage_differs │ make_differs │ fuel_differs │ completed_date_differs │
│  int64  │       int64       │       int64        │       int64       │     int64      │      int64      │    int64     │    int64     │         int64          │
├─────────┼───────────────────┼────────────────────┼───────────────────┼────────────────┼─────────────────┼──────────────┼──────────────┼────────────────────────┤
│ 3783238 │           1171839 │                777 │                 0 │              0 │            2038 │          288 │         2267 │                      0 │
└─────────┴───────────────────┴────────────────────┴───────────────────┴────────────────┴─────────────────┴──────────────┴──────────────┴────────────────────────┘



The results above show that ~5,400 test IDs are repeated with diffs in these categories.

Worst offenders list:

In [44]:
con.sql(f"""
    WITH t AS (
        SELECT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT test_id,
           COUNT(*)                   AS row_count,
           COUNT(DISTINCT vehicle_id) AS vehicles,
           COUNT(DISTINCT make)       AS makes
    FROM t
    GROUP BY test_id
    ORDER BY row_count DESC
    LIMIT 10
""").show()

┌────────────┬───────────┬──────────┬───────┐
│  test_id   │ row_count │ vehicles │ makes │
│  varchar   │   int64   │  int64   │ int64 │
├────────────┼───────────┼──────────┼───────┤
│ 1586552547 │      1517 │        1 │     1 │
│ 1502694417 │       441 │        1 │     1 │
│ 905868655  │       308 │        1 │     1 │
│ 1825417585 │       252 │        1 │     1 │
│ 1677208933 │       240 │        1 │     1 │
│ 962770155  │       221 │        1 │     1 │
│ 725737941  │       176 │        1 │     1 │
│ 350937541  │       154 │        1 │     1 │
│ 1654835091 │       144 │        1 │     1 │
│ 311242945  │       130 │        1 │     1 │
└────────────┴───────────┴──────────┴───────┘
  10 rows                         4 columns



This is the same grouping idea as before. ORDER BY row_count DESC puts the biggest first, and LIMIT 10 keeps the top ten.
Read it like this: if vehicles is also large for the top rows, one test ID is being shared by many vehicles. If vehicles is 1 even for tests with hundreds of rows, those rows are copies of one test.

New variable: ITEM_JAN created for failure items.

In [45]:
ITEM_JAN = r"""read_csv(
    '../data/raw/MOT Testing data failure item (2024)/test_item_202401.csv',
    all_varchar=true,
    quote='"',
    escape='\'
)"""

In [46]:
con.sql(f"SELECT * FROM {ITEM_JAN} LIMIT 5").show()

┌────────────┬─────────┬───────────────┬───────────────────────────────┬────────────────┬──────────────────────────┐
│  test_id   │ rfr_id  │ rfr_type_code │ mot_test_rfr_location_type_id │ dangerous_mark │      completed_date      │
│  varchar   │ varchar │    varchar    │            varchar            │    varchar     │         varchar          │
├────────────┼─────────┼───────────────┼───────────────────────────────┼────────────────┼──────────────────────────┤
│ 1479190053 │ 31361   │ A             │ 9                             │ NULL           │ 2024-01-01T11:59:33.000Z │
│ 1479190053 │ 30335   │ A             │ 7                             │ NULL           │ 2024-01-01T11:59:33.000Z │
│ 1479190053 │ 32017   │ P             │ 25                            │ NULL           │ 2024-01-01T11:59:33.000Z │
│ 1479190053 │ 30202   │ A             │ 1                             │ NULL           │ 2024-01-01T11:59:33.000Z │
│ 1479190053 │ 30335   │ A             │ 9                      

Find the biggest cuplrits for copies in the failure items data. Repeats here are much larger.

In [47]:
con.sql(f"""
    SELECT test_id, COUNT(*) AS item_rows
    FROM {ITEM_JAN}
    WHERE test_id IN ('1586552547', '1502694417', '905868655', '1825417585', '1677208933')
    GROUP BY test_id
    ORDER BY item_rows DESC
""").show()

┌────────────┬───────────┐
│  test_id   │ item_rows │
│  varchar   │   int64   │
├────────────┼───────────┤
│ 1586552547 │      9583 │
│ 1677208933 │      1920 │
│ 905868655  │       189 │
│ 1825417585 │       144 │
└────────────┴───────────┘



Now, let's see if the repeats are identical between JAN and ITEM_JAN:

In [48]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           (SELECT COUNT(*) FROM (
                SELECT DISTINCT * FROM {JAN} WHERE test_id = '1586552547'
           )) AS distinct_rows
    FROM {JAN}
    WHERE test_id = '1586552547'
""").show()

┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│       1517 │             1 │
└────────────┴───────────────┘



In [49]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           (SELECT COUNT(*) FROM (
                SELECT DISTINCT * FROM {ITEM_JAN} WHERE test_id = '1586552547'
           )) AS distinct_rows
    FROM {ITEM_JAN}
    WHERE test_id = '1586552547'
""").show()

┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│       9583 │             6 │
└────────────┴───────────────┘



The outer query counts every row for that test. The inner SELECT DISTINCT * counts only the different rows.

distinct_rows = 1 means all copies are identical. A small number means a few real values repeated many times. A number close to total_rows means the rows are genuinely different.

The very first peek showed five identical rows for a Porsche 911 pass (test 886090729).

Let’s see how many defect items it has:

In [50]:
con.sql(f"""
    SELECT 'results file' AS source, COUNT(*) AS n
    FROM {JAN} WHERE test_id = '886090729'
    UNION ALL
    SELECT 'items file', COUNT(*)
    FROM {ITEM_JAN} WHERE test_id = '886090729'
""").show()

┌──────────────┬───────┐
│    source    │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ results file │    12 │
│ items file   │     0 │
└──────────────┴───────┘



UNION ALL stacks the two results into one table, with a label saying which file each came from.

Results file, worst test: 1,517 rows but only 1 distinct row. Those are pure copies.

Items file, same test: 9,583 rows but only 6 distinct rows. The same six real defects are repeated over and over.

Porsche test 886090729: 12 rows in the results file but 0 items, so it’s a pass with no defects, and it’s still repeated 12 times.

In [51]:
con.sql(f"""
    WITH d AS (
        SELECT DISTINCT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT COUNT(*) AS rows_after_dedupe,
           COUNT(DISTINCT test_id) AS distinct_tests,
           (SELECT COUNT(*)
            FROM (SELECT test_id FROM d GROUP BY test_id HAVING COUNT(*) > 1)
           ) AS tests_with_conflicts
    FROM d
""").show()

┌───────────────────┬────────────────┬──────────────────────┐
│ rows_after_dedupe │ distinct_tests │ tests_with_conflicts │
│       int64       │     int64      │        int64         │
├───────────────────┼────────────────┼──────────────────────┤
│           3848449 │        3783238 │                64159 │
└───────────────────┴────────────────┴──────────────────────┘



d is January with the repeated headers and the exact copies removed (DISTINCT *).

HAVING COUNT(*) > 1 is like WHERE, but it filters after grouping, so it keeps only the test IDs that still have more than one row.
You should see rows_after_dedupe of 3,848,449 and distinct_tests of 3,783,238, matching the earlier numbers. tests_with_conflicts is the number missing.

In [52]:
con.sql(f"""
    WITH d AS (
        SELECT DISTINCT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT *
    FROM d
    WHERE test_id IN (
        SELECT test_id FROM d GROUP BY test_id HAVING COUNT(*) > 1
    )
    ORDER BY test_id
    LIMIT 20
""").show(max_width=300)

┌────────────┬────────────┬────────────┬───────────────┬───────────┬─────────────┬──────────────┬───────────────┬──────────┬──────────┬─────────┬───────────┬───────────────────┬────────────────┬──────────────────────────┐
│  test_id   │ vehicle_id │ test_date  │ test_class_id │ test_type │ test_result │ test_mileage │ postcode_area │   make   │  model   │ colour  │ fuel_type │ cylinder_capacity │ first_use_date │      completed_date      │
│  varchar   │  varchar   │  varchar   │    varchar    │  varchar  │   varchar   │   varchar    │    varchar    │ varchar  │ varchar  │ varchar │  varchar  │      varchar      │    varchar     │         varchar          │
├────────────┼────────────┼────────────┼───────────────┼───────────┼─────────────┼──────────────┼───────────────┼──────────┼──────────┼─────────┼───────────┼───────────────────┼────────────────┼──────────────────────────┤
│ 1000007803 │ 347452908  │ 2024-01-09 │ 4             │ RT        │ P           │ 217676.0     │ CM            

The inner SELECT finds the conflicting test IDs, and the outer one pulls all the columns for those tests.

ORDER BY test_id puts the rows for each test next to each other, so you can spot which column differs.

max_width=300 stops DuckDB from truncating the wide table with ....

Pattern | Example from the output | Matters?

first_use_date differs within the same year
1999-09-01 vs 1999-12-31, 1996-04-18 vs 1996-12-30, 1991-12-31 vs 1991-09-13
Slightly: vehicle age could shift by a year

test_class_id differs
Vauxhall Vivaro as class 7 and class 4; Honda NSS as class 1 and class 2
Yes: our scope filter is class 4

cylinder_capacity is NULL vs 0
Nissan Leaf (electric, so no engine size)
No: both mean “no engine capacity”

colour differs
Porsche 911 as BLUE and GREY
Not for our questions

Check against all 15 columns.

In [53]:
con.sql(f"""
    WITH d AS (
        SELECT DISTINCT * FROM {JAN} WHERE test_id <> 'test_id'
    )
    SELECT
        COUNT(*) AS conflicting_tests,
        COUNT(*) FILTER (WHERE n_vehicle > 1)   AS vehicle_id,
        COUNT(*) FILTER (WHERE n_date > 1)      AS test_date,
        COUNT(*) FILTER (WHERE n_class > 1)     AS test_class_id,
        COUNT(*) FILTER (WHERE n_type > 1)      AS test_type,
        COUNT(*) FILTER (WHERE n_result > 1)    AS test_result,
        COUNT(*) FILTER (WHERE n_mileage > 1)   AS test_mileage,
        COUNT(*) FILTER (WHERE n_postcode > 1)  AS postcode_area,
        COUNT(*) FILTER (WHERE n_make > 1)      AS make,
        COUNT(*) FILTER (WHERE n_model > 1)     AS model,
        COUNT(*) FILTER (WHERE n_colour > 1)    AS colour,
        COUNT(*) FILTER (WHERE n_fuel > 1)      AS fuel_type,
        COUNT(*) FILTER (WHERE n_cc > 1)        AS cylinder_capacity,
        COUNT(*) FILTER (WHERE n_first > 1)     AS first_use_date,
        COUNT(*) FILTER (WHERE n_completed > 1) AS completed_date
    FROM (
        SELECT test_id,
               COUNT(DISTINCT vehicle_id)        AS n_vehicle,
               COUNT(DISTINCT test_date)         AS n_date,
               COUNT(DISTINCT test_class_id)     AS n_class,
               COUNT(DISTINCT test_type)         AS n_type,
               COUNT(DISTINCT test_result)       AS n_result,
               COUNT(DISTINCT test_mileage)      AS n_mileage,
               COUNT(DISTINCT postcode_area)     AS n_postcode,
               COUNT(DISTINCT make)              AS n_make,
               COUNT(DISTINCT model)             AS n_model,
               COUNT(DISTINCT colour)            AS n_colour,
               COUNT(DISTINCT fuel_type)         AS n_fuel,
               COUNT(DISTINCT cylinder_capacity) AS n_cc,
               COUNT(DISTINCT first_use_date)    AS n_first,
               COUNT(DISTINCT completed_date)    AS n_completed
        FROM d
        GROUP BY test_id
        HAVING COUNT(*) > 1
    )
""").show(max_width=300)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────────┬───────────┬───────────────┬───────────┬─────────────┬──────────────┬───────────────┬───────┬───────┬────────┬───────────┬───────────────────┬────────────────┬────────────────┐
│ conflicting_tests │ vehicle_id │ test_date │ test_class_id │ test_type │ test_result │ test_mileage │ postcode_area │ make  │ model │ colour │ fuel_type │ cylinder_capacity │ first_use_date │ completed_date │
│       int64       │   int64    │   int64   │     int64     │   int64   │    int64    │    int64     │     int64     │ int64 │ int64 │ int64  │   int64   │       int64       │     int64      │     int64      │
├───────────────────┼────────────┼───────────┼───────────────┼───────────┼─────────────┼──────────────┼───────────────┼───────┼───────┼────────┼───────────┼───────────────────┼────────────────┼────────────────┤
│             64159 │        777 │         0 │          7952 │         0 │           0 │         2038 │           343 │   288 │  2224 │   8181 │      2267 │

First_use_date shows massive differences. From the tables above, the years align but the dates might differ.

In [54]:
con.sql(f"""
    SELECT fuel_type, COUNT(*) AS n
    FROM {JAN}
    WHERE test_id <> 'test_id'
    GROUP BY fuel_type
    ORDER BY n DESC
""").show()

┌─────────────────────────┬─────────┐
│        fuel_type        │    n    │
│         varchar         │  int64  │
├─────────────────────────┼─────────┤
│ PE                      │ 3057610 │
│ DI                      │ 2844800 │
│ HY                      │  132557 │
│ Hybrid Electric (Clean) │   44898 │
│ EL                      │   39317 │
│ ED                      │    9368 │
│ Electric                │    8701 │
│ OT                      │    1619 │
│ LP                      │    1432 │
│ GB                      │     383 │
│ FC                      │      30 │
│ GA                      │      29 │
│ GD                      │      25 │
│ CN                      │      10 │
│ ST                      │       9 │
└─────────────────────────┴─────────┘
  15 rows                 2 columns



The above results show fuel type is recorded differently, when meaning the same thing.

In [55]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW jan_norm AS
    SELECT DISTINCT
        * EXCLUDE (fuel_type, first_use_date, cylinder_capacity, colour, completed_date),
        CASE fuel_type
            WHEN 'Hybrid Electric (Clean)' THEN 'HY'
            WHEN 'Electric' THEN 'EL'
            ELSE fuel_type
        END AS fuel_type,
        NULLIF(NULLIF(cylinder_capacity, '0'), 'NULL') AS cylinder_capacity,
        TRY_CAST(LEFT(first_use_date, 4) AS INTEGER) AS first_use_year
    FROM {JAN}
    WHERE test_id <> 'test_id'
""")

con.execute(...) is for commands that do something (like creating a view), while con.sql(...) is for queries you want to look at. It prints nothing when it works.

TEMP means the view disappears when you close the notebook’s connection. That’s fine for exploring.

'* EXCLUDE (...) means “all columns except these”, so we can then re-add the ones we’re rewriting.

The CASE turns the two long fuel names into their codes and leaves everything else alone.

NULLIF(x, '0') returns NULL when x equals '0'. I nested a second NULLIF for the text 'NULL', because I can’t tell from the output whether your files contain a true empty value or the literal word NULL. This handles both.

LEFT(first_use_date, 4) takes the first four characters (the year), and TRY_CAST(... AS INTEGER) turns it into a number.

SELECT DISTINCT goes last in effect: it removes duplicates after the rewrites, which is the whole point.

In [56]:
con.sql("""
    SELECT
        COUNT(*) AS conflicting_tests,
        COUNT(*) FILTER (WHERE n_vehicle > 1)  AS vehicle_id,
        COUNT(*) FILTER (WHERE n_class > 1)    AS test_class_id,
        COUNT(*) FILTER (WHERE n_mileage > 1)  AS test_mileage,
        COUNT(*) FILTER (WHERE n_postcode > 1) AS postcode_area,
        COUNT(*) FILTER (WHERE n_make > 1)     AS make,
        COUNT(*) FILTER (WHERE n_model > 1)    AS model,
        COUNT(*) FILTER (WHERE n_fuel > 1)     AS fuel_type,
        COUNT(*) FILTER (WHERE n_cc > 1)       AS cylinder_capacity,
        COUNT(*) FILTER (WHERE n_first > 1)    AS first_use_year
    FROM (
        SELECT test_id,
               COUNT(DISTINCT vehicle_id)        AS n_vehicle,
               COUNT(DISTINCT test_class_id)     AS n_class,
               COUNT(DISTINCT test_mileage)      AS n_mileage,
               COUNT(DISTINCT postcode_area)     AS n_postcode,
               COUNT(DISTINCT make)              AS n_make,
               COUNT(DISTINCT model)             AS n_model,
               COUNT(DISTINCT fuel_type)         AS n_fuel,
               COUNT(DISTINCT cylinder_capacity) AS n_cc,
               COUNT(DISTINCT first_use_year)    AS n_first
        FROM jan_norm
        GROUP BY test_id
        HAVING COUNT(*) > 1
    )
""").show(max_width=300)

┌───────────────────┬────────────┬───────────────┬──────────────┬───────────────┬───────┬───────┬───────────┬───────────────────┬────────────────┐
│ conflicting_tests │ vehicle_id │ test_class_id │ test_mileage │ postcode_area │ make  │ model │ fuel_type │ cylinder_capacity │ first_use_year │
│       int64       │   int64    │     int64     │    int64     │     int64     │ int64 │ int64 │   int64   │       int64       │     int64      │
├───────────────────┼────────────┼───────────────┼──────────────┼───────────────┼───────┼───────┼───────────┼───────────────────┼────────────────┤
│             16270 │        777 │          7952 │         2038 │           343 │   288 │  2224 │      2249 │               811 │           1224 │
└───────────────────┴────────────┴───────────────┴──────────────┴───────────────┴───────┴───────┴───────────┴───────────────────┴────────────────┘



Conflicting tests dropped from 64,159 to 16,270, a 75% cut, which is about 0.43% of January’s 3.78 million tests.

Do conflicting tests fail differently?:

In [57]:
con.sql("""
    WITH t AS (
        SELECT test_id,
               COUNT(*)               AS versions,
               ANY_VALUE(test_type)   AS test_type,
               ANY_VALUE(test_result) AS test_result
        FROM jan_norm
        GROUP BY test_id
    )
    SELECT versions > 1 AS has_conflict,
           COUNT(*) AS tests,
           ROUND(100.0 * COUNT(*) FILTER (WHERE test_result IN ('F', 'PRS')) / COUNT(*), 1)
               AS pct_initial_fail
    FROM t
    WHERE test_type = 'NT' AND test_result IN ('P', 'F', 'PRS')
    GROUP BY has_conflict
    ORDER BY has_conflict
""").show()

┌──────────────┬─────────┬──────────────────┐
│ has_conflict │  tests  │ pct_initial_fail │
│   boolean    │  int64  │      double      │
├──────────────┼─────────┼──────────────────┤
│ false        │ 3098266 │             30.0 │
│ true         │   12862 │             29.4 │
└──────────────┴─────────┴──────────────────┘



The inner query gives one row per test with how many versions of it exist (versions). ANY_VALUE picks one value from the group, which is safe here because test_type and test_result never differ between copies.

The outer query splits tests into two groups, versions > 1 (conflicting) and not, and calculates the share of initial fails in each, using the guide’s definition (F or PRS).

The WHERE keeps only initial tests with a clear outcome.

The rates are 0.6 percentage points apart. With about 12,900 tests in the smaller group, a gap that size is within normal random variation (roughly one and a half standard errors). So excluding them shouldn’t noticeably skew the overall failure rate.

Create table and load January:

In [90]:
con.execute("""
    CREATE OR REPLACE TABLE results_dedup (
        test_id           VARCHAR,
        vehicle_id        VARCHAR,
        test_date         VARCHAR,
        test_class_id     VARCHAR,
        test_type         VARCHAR,
        test_result       VARCHAR,
        test_mileage      VARCHAR,
        postcode_area     VARCHAR,
        make              VARCHAR,
        model             VARCHAR,
        fuel_type         VARCHAR,
        cylinder_capacity VARCHAR,
        first_use_year    INTEGER,
        filename          VARCHAR
    )
""")

Each line is a column name followed by its data type. VARCHAR is text, and INTEGER is a whole number. We keep most columns as text for now and convert them in the final clean step, so a bad value can’t stop the load.

CREATE OR REPLACE means re-running this cell wipes the table and starts empty. That matters for Cell 3 below.

filename will record which monthly file each row came from, so we can investigate cross-file duplicates later.

Define a function that loads one monthly file:

In [91]:
def load_month(path):
    con.execute(rf"""
        INSERT INTO results_dedup BY NAME
        SELECT DISTINCT
            * EXCLUDE (fuel_type, first_use_date, cylinder_capacity, colour, completed_date),
            CASE fuel_type
                WHEN 'Hybrid Electric (Clean)' THEN 'HY'
                WHEN 'Electric' THEN 'EL'
                ELSE fuel_type
            END AS fuel_type,
            NULLIF(NULLIF(cylinder_capacity, '0'), 'NULL') AS cylinder_capacity,
            TRY_CAST(LEFT(first_use_date, 4) AS INTEGER) AS first_use_year
        FROM read_csv('{path}', all_varchar=true, quote='"', escape='\', filename=true)
        WHERE test_id <> 'test_id'
    """)

def load_month(path): defines a function called load_month that takes one input, path, the location of a file. Everything indented underneath belongs to it.

Inside, the SQL is the view you already wrote, with three changes. It’s now INSERT INTO results_dedup, which adds rows to the table. BY NAME matches columns by name rather than position, so the order doesn’t matter. And filename=true adds the filename column.

The rf""" prefix combines two things you’ve seen separately: r keeps the backslash in escape='\' as typed, and f swaps {path} for the file location you pass in.

### Run for January

In [60]:
jan_path = "../data/raw/MOT testing data results (2024)/test_result_202401.csv"
load_month(jan_path)

In [61]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM results_dedup)                 AS table_rows,
           (SELECT COUNT(*) FROM jan_norm)                      AS view_rows,
           (SELECT COUNT(DISTINCT test_id) FROM results_dedup)  AS distinct_tests
""").show()

┌────────────┬───────────┬────────────────┐
│ table_rows │ view_rows │ distinct_tests │
│   int64    │   int64   │     int64      │
├────────────┼───────────┼────────────────┤
│    3799673 │   3799673 │        3783238 │
└────────────┴───────────┴────────────────┘



### Sort and add for all months

In [92]:
results_folder = "../data/raw/MOT testing data results (2024)"
files = sorted(f for f in os.listdir(results_folder) if f.endswith(".csv"))
files

['test_result_202401.csv',
 'test_result_202402.csv',
 'test_result_202403.csv',
 'test_result_202404.csv',
 'test_result_202405.csv',
 'test_result_202406.csv',
 'test_result_202407.csv',
 'test_result_202408.csv',
 'test_result_202409.csv',
 'test_result_202410.csv',
 'test_result_202411.csv',
 'test_result_202412.csv']

In [93]:
import time

for f in files:
    start = time.time()
    load_month(f"{results_folder}/{f}")
    n = con.sql("SELECT COUNT(*) FROM results_dedup").fetchall()[0][0]
    print(f"{f}: table now has {n:,} rows ({time.time() - start:.0f}s)")

test_result_202401.csv: table now has 3,799,673 rows (1s)
test_result_202402.csv: table now has 7,416,169 rows (1s)
test_result_202403.csv: table now has 11,486,638 rows (1s)
test_result_202404.csv: table now has 14,620,585 rows (1s)
test_result_202405.csv: table now has 17,750,534 rows (1s)
test_result_202406.csv: table now has 21,037,179 rows (1s)
test_result_202407.csv: table now has 24,653,251 rows (1s)
test_result_202408.csv: table now has 28,173,688 rows (1s)
test_result_202409.csv: table now has 32,186,123 rows (1s)
test_result_202410.csv: table now has 36,418,360 rows (1s)
test_result_202411.csv: table now has 40,104,799 rows (1s)
test_result_202412.csv: table now has 42,767,586 rows (0s)


Check all totals:

In [94]:
con.sql("""
    SELECT COUNT(*) AS table_rows,
           COUNT(DISTINCT test_id) AS distinct_tests,
           COUNT(DISTINCT filename) AS files
    FROM results_dedup
""").show()

┌────────────┬────────────────┬───────┐
│ table_rows │ distinct_tests │ files │
│   int64    │     int64      │ int64 │
├────────────┼────────────────┼───────┤
│   42767586 │       42598624 │    12 │
└────────────┴────────────────┴───────┘



Any tests in multiple CSV files?

In [95]:
con.sql("""
    SELECT COUNT(*) AS tests_in_multiple_files
    FROM (
        SELECT test_id
        FROM results_dedup
        GROUP BY test_id
        HAVING COUNT(DISTINCT filename) > 1
    )
""").show()

┌─────────────────────────┐
│ tests_in_multiple_files │
│          int64          │
├─────────────────────────┤
│                       0 │
└─────────────────────────┘



In [96]:
con.sql("""
    SELECT filename, COUNT(*) AS n
    FROM results_dedup
    GROUP BY filename
    ORDER BY filename
""").show()

┌────────────────────────────────────────────────────────────────────┬─────────┐
│                              filename                              │    n    │
│                              varchar                               │  int64  │
├────────────────────────────────────────────────────────────────────┼─────────┤
│ ../data/raw/MOT testing data results (2024)/test_result_202401.csv │ 3799673 │
│ ../data/raw/MOT testing data results (2024)/test_result_202402.csv │ 3616496 │
│ ../data/raw/MOT testing data results (2024)/test_result_202403.csv │ 4070469 │
│ ../data/raw/MOT testing data results (2024)/test_result_202404.csv │ 3133947 │
│ ../data/raw/MOT testing data results (2024)/test_result_202405.csv │ 3129949 │
│ ../data/raw/MOT testing data results (2024)/test_result_202406.csv │ 3286645 │
│ ../data/raw/MOT testing data results (2024)/test_result_202407.csv │ 3616072 │
│ ../data/raw/MOT testing data results (2024)/test_result_202408.csv │ 3520437 │
│ ../data/raw/MOT testing da

The monthly counts look plausible. March, September and October are the busiest months. That fits the UK’s yearly plate changes, since a car’s first MOT is due three years after registration.

Record all conflictd

In [97]:
con.execute("""
    CREATE OR REPLACE TABLE test_conflicts AS
    SELECT test_id, COUNT(*) AS versions
    FROM results_dedup
    GROUP BY test_id
    HAVING COUNT(*) > 1
""")

In [98]:
con.sql("""
    SELECT COUNT(*)                    AS conflicting_tests,
           SUM(versions)               AS rows_involved,
           SUM(versions) - COUNT(*)    AS extra_rows
    FROM test_conflicts
""").show()

┌───────────────────┬───────────────┬────────────┐
│ conflicting_tests │ rows_involved │ extra_rows │
│       int64       │    int128     │   int128   │
├───────────────────┼───────────────┼────────────┤
│            167725 │        336687 │     168962 │
└───────────────────┴───────────────┴────────────┘



168k matches the value calculated earlier. ~0.4% of total tests.

### Build the clean table

In [99]:
con.execute("""
    CREATE OR REPLACE TABLE test_result_clean AS
    SELECT
        TRY_CAST(test_id AS BIGINT)                          AS test_id,
        TRY_CAST(vehicle_id AS BIGINT)                       AS vehicle_id,
        TRY_CAST(test_date AS DATE)                          AS test_date,
        test_class_id,
        test_type,
        test_result,
        TRY_CAST(TRY_CAST(test_mileage AS DOUBLE) AS BIGINT) AS test_mileage,
        postcode_area,
        UPPER(TRIM(make))                                    AS make,
        UPPER(TRIM(model))                                   AS model,
        fuel_type,
        TRY_CAST(cylinder_capacity AS INTEGER)               AS cylinder_capacity,
        first_use_year
    FROM results_dedup
    WHERE test_id NOT IN (SELECT test_id FROM test_conflicts)
""")

TRY_CAST(x AS BIGINT) converts text to a whole number, or gives NULL if it can’t, so a bad value can’t crash the build.

Mileage has two casts because it’s stored as text like 71614.0. We go text → decimal number → whole number.

TRY_CAST(test_date AS DATE) works directly because the dates are already ISO format.

UPPER(TRIM(make)) removes stray spaces and makes the case consistent, so Ford and FORD can’t be counted as different makes.

The class, type, result, postcode and fuel columns stay as text codes.

WHERE test_id NOT IN (SELECT test_id FROM test_conflicts) keeps every test that is not in the conflicts table. That’s the exclusion rule.

Check the clean table:

In [100]:
con.sql("""
    SELECT COUNT(*)                                       AS clean_rows,
           COUNT(DISTINCT test_id)                        AS clean_tests,
           COUNT(*) FILTER (WHERE test_id IS NULL)        AS null_test_id,
           COUNT(*) FILTER (WHERE test_date IS NULL)      AS null_test_date,
           COUNT(*) FILTER (WHERE test_mileage IS NULL)   AS null_mileage
    FROM test_result_clean
""").show()

┌────────────┬─────────────┬──────────────┬────────────────┬──────────────┐
│ clean_rows │ clean_tests │ null_test_id │ null_test_date │ null_mileage │
│   int64    │    int64    │    int64     │     int64      │    int64     │
├────────────┼─────────────┼──────────────┼────────────────┼──────────────┤
│   42430899 │    42430899 │            0 │              0 │       321853 │
└────────────┴─────────────┴──────────────┴────────────────┴──────────────┘



Uniqueness is fixed. clean_rows equals clean_tests (42,430,899), so every test now appears exactly once.

The totals reconcile. 42,430,899 clean tests plus 167,725 conflicting tests is 42,598,624, your all-files distinct count. Every test is accounted for, either kept or excluded.

The exclusion check matches. extra_rows is 168,962, the gap calculated.

No nulls in test_id or test_date. The 321,853 null mileages (0.76%) are expected, since it includes blank readings.

A correction. I guessed about 190,000 conflicting tests for the year. The real figure is 167,725 (0.39%), a bit lower than January’s 0.43%. 

Nearly all conflicts are just two versions. 336,687 rows across 167,725 tests averages about 2.007 rows per test.

In [101]:
con.sql("""
    SELECT 'test_result' AS col, test_result AS value, COUNT(*) AS n
    FROM test_result_clean GROUP BY test_result
    UNION ALL
    SELECT 'test_type', test_type, COUNT(*) FROM test_result_clean GROUP BY test_type
    UNION ALL
    SELECT 'test_class_id', test_class_id, COUNT(*) FROM test_result_clean GROUP BY test_class_id
    UNION ALL
    SELECT 'fuel_type', fuel_type, COUNT(*) FROM test_result_clean GROUP BY fuel_type
    ORDER BY col, n DESC
""").show(max_rows=100)

┌───────────────┬─────────┬──────────┐
│      col      │  value  │    n     │
│    varchar    │ varchar │  int64   │
├───────────────┼─────────┼──────────┤
│ fuel_type     │ PE      │ 22300120 │
│ fuel_type     │ DI      │ 18066134 │
│ fuel_type     │ HY      │  1459511 │
│ fuel_type     │ EL      │   507420 │
│ fuel_type     │ ED      │    76215 │
│ fuel_type     │ OT      │     8352 │
│ fuel_type     │ LP      │     8103 │
│ fuel_type     │ GB      │     4429 │
│ fuel_type     │ GA      │      222 │
│ fuel_type     │ FC      │      166 │
│ fuel_type     │ GD      │      129 │
│ fuel_type     │ CN      │       68 │
│ fuel_type     │ LN      │       22 │
│ fuel_type     │ ST      │        8 │
│ test_class_id │ 4       │ 40025025 │
│ test_class_id │ 7       │  1299625 │
│ test_class_id │ 2       │   811592 │
│ test_class_id │ 1       │   235913 │
│ test_class_id │ 5       │    48870 │
│ test_class_id │ 3       │     9874 │
│ test_result   │ P       │ 32310599 │
│ test_result   │ F      

Each SELECT counts how often each value appears in one column. UNION ALL stacks the four results into one table, with the first column naming which field each row belongs to.

ORDER BY col, n DESC groups the rows by field and puts the most common values first.

max_rows=100 stops DuckDB from hiding rows in the middle of a long table.

Checking whether the values match the guide: result codes P, F, PRS, ABA, ABR; types NT, RT and so on; classes 0 to 7; and the fuel codes. Anything unexpected is worth noting.

Edge cases and known changes:

In [102]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE test_mileage = 0)                  AS zero_mileage,
        COUNT(*) FILTER (WHERE test_mileage > 500000)             AS over_500k,
        COUNT(*) FILTER (WHERE test_mileage > 1000000)            AS over_1m,
        MEDIAN(test_mileage) FILTER (WHERE test_mileage > 0)      AS median_mileage,
        COUNT(*) FILTER (WHERE first_use_year = 1971)             AS year_1971,
        COUNT(*) FILTER (WHERE first_use_year IS NULL)            AS null_first_use_year,
        COUNT(*) FILTER (WHERE first_use_year > YEAR(test_date))  AS first_use_after_test,
        COUNT(*) FILTER (WHERE postcode_area = 'XX')              AS postcode_xx,
        COUNT(*) FILTER (WHERE postcode_area IS NULL)             AS postcode_null,
        COUNT(*) FILTER (WHERE make = 'UNCLASSIFIED'
                            OR model = 'UNCLASSIFIED')            AS unclassified,
        MIN(test_date)                                            AS earliest_test,
        MAX(test_date)                                            AS latest_test
    FROM test_result_clean
""").show(max_width=300)

┌──────────────┬───────────┬─────────┬────────────────┬───────────┬─────────────────────┬──────────────────────┬─────────────┬───────────────┬──────────────┬───────────────┬─────────────┐
│ zero_mileage │ over_500k │ over_1m │ median_mileage │ year_1971 │ null_first_use_year │ first_use_after_test │ postcode_xx │ postcode_null │ unclassified │ earliest_test │ latest_test │
│    int64     │   int64   │  int64  │     double     │   int64   │        int64        │        int64         │    int64    │     int64     │    int64     │     date      │    date     │
├──────────────┼───────────┼─────────┼────────────────┼───────────┼─────────────────────┼──────────────────────┼─────────────┼───────────────┼──────────────┼───────────────┼─────────────┤
│            0 │      5393 │       0 │        68681.0 │      3049 │                   0 │                    2 │        2839 │             0 │       108292 │ 2024-01-01    │ 2024-12-30  │
└──────────────┴───────────┴─────────┴────────────────┴─────

Mileage: how many zeros (no reading), how many are above 500,000 and 1,000,000 miles, and the median of the real readings.

First-use year: how many are the 1971 placeholder, how many are missing, and how many are impossible (first use after the test year).

Postcode and make/model: how many are XX, how many are NULL, and how many are labelled UNCLASSIFIED.

Dates: the earliest and latest test date, which should fall inside 2024.

### How high does the mileage go?

In [103]:
con.sql("""
    SELECT MIN(test_mileage)                    AS min_mileage,
           QUANTILE_CONT(test_mileage, 0.5)     AS median,
           QUANTILE_CONT(test_mileage, 0.99)    AS p99,
           QUANTILE_CONT(test_mileage, 0.999)   AS p999,
           MAX(test_mileage)                    AS max_mileage
    FROM test_result_clean
    WHERE test_class_id = '4'
""").show()

┌─────────────┬─────────┬──────────┬──────────────────┬─────────────┐
│ min_mileage │ median  │   p99    │       p999       │ max_mileage │
│    int64    │ double  │  double  │      double      │    int64    │
├─────────────┼─────────┼──────────┼──────────────────┼─────────────┤
│           1 │ 69318.0 │ 214649.0 │ 313484.438000001 │      999999 │
└─────────────┴─────────┴──────────┴──────────────────┴─────────────┘



In [104]:
con.sql("""
    SELECT test_id, make, model, test_mileage, first_use_year
    FROM test_result_clean
    WHERE test_class_id = '4' AND test_mileage IS NOT NULL
    ORDER BY test_mileage DESC
    LIMIT 10
""").show()

┌────────────┬───────────────┬─────────┬──────────────┬────────────────┐
│  test_id   │     make      │  model  │ test_mileage │ first_use_year │
│   int64    │    varchar    │ varchar │    int64     │     int32      │
├────────────┼───────────────┼─────────┼──────────────┼────────────────┤
│ 1665872099 │ AUDI          │ TT      │       999999 │           2003 │
│  629734645 │ AUDI          │ A3      │       999999 │           2002 │
│  838472711 │ ROVER         │ 75      │       999999 │           2004 │
│  274574345 │ MERCEDES-BENZ │ CLS     │       999999 │           2007 │
│ 1360466099 │ MG            │ ZT      │       999999 │           2004 │
│  830678913 │ AUDI          │ A3      │       999999 │           2003 │
│  264668465 │ FORD          │ TRANSIT │       999999 │           2010 │
│  665744771 │ VAUXHALL      │ ASTRA   │       999999 │           2006 │
│   36727951 │ ROVER         │ 75      │       999999 │           2001 │
│ 1914913581 │ AUDI          │ TT      │       9999

QUANTILE_CONT(x, 0.99) gives the value that 99% of the readings fall below. Comparing the median, the 99th and the 99.9th percentile with the maximum shows how long the tail is.

The second query lists the ten highest readings with their vehicles, so you can judge whether they’re believable (a high-mileage taxi) or typos (a 15-year-old Fiesta at 900,000 miles).

Rebuild clean table:

In [105]:
con.execute("""
    CREATE OR REPLACE TABLE test_result_clean AS
    SELECT
        TRY_CAST(test_id AS BIGINT)                          AS test_id,
        TRY_CAST(vehicle_id AS BIGINT)                       AS vehicle_id,
        TRY_CAST(test_date AS DATE)                          AS test_date,
        test_class_id,
        test_type,
        test_result,
        CASE WHEN test_result IN ('P', 'F', 'PRS')
             THEN test_result IN ('F', 'PRS') END            AS is_initial_fail,
        TRY_CAST(TRY_CAST(test_mileage AS DOUBLE) AS BIGINT) AS test_mileage,
        NULLIF(postcode_area, 'XX')                          AS postcode_area,
        UPPER(TRIM(make))                                    AS make,
        UPPER(TRIM(model))                                   AS model,
        (UPPER(TRIM(make)) = 'UNCLASSIFIED'
         OR UPPER(TRIM(model)) = 'UNCLASSIFIED')             AS is_unclassified,
        fuel_type,
        CASE fuel_type
            WHEN 'PE' THEN 'Petrol'
            WHEN 'DI' THEN 'Diesel'
            WHEN 'HY' THEN 'Hybrid'
            WHEN 'EL' THEN 'Electric'
            WHEN 'ED' THEN 'Electric Diesel'
            WHEN 'LP' THEN 'LPG'
            WHEN 'GB' THEN 'Gas Bi-Fuel'
            WHEN 'GA' THEN 'Gas'
            WHEN 'GD' THEN 'Gas Diesel'
            WHEN 'FC' THEN 'Fuel Cell'
            WHEN 'CN' THEN 'CNG'
            WHEN 'LN' THEN 'LNG'
            WHEN 'ST' THEN 'Steam'
            ELSE 'Other'
        END                                                  AS fuel_type_desc,
        TRY_CAST(cylinder_capacity AS INTEGER)               AS cylinder_capacity,
        NULLIF(first_use_year, 1971)                         AS first_use_year,
        CASE WHEN first_use_year <> 1971
              AND first_use_year <= YEAR(TRY_CAST(test_date AS DATE))
             THEN YEAR(TRY_CAST(test_date AS DATE)) - first_use_year
        END                                                  AS vehicle_age_years
    FROM results_dedup
    WHERE test_id NOT IN (SELECT test_id FROM test_conflicts)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_initial_fail: true for F or PRS, false for P, and left NULL for aborted or abandoned tests, because they have no pass or fail outcome. 
The CASE WHEN ... THEN ... END means “only work this out for P, F or PRS rows, otherwise leave it empty”.

NULLIF(postcode_area, 'XX'): returns NULL when the value is XX, and otherwise leaves it alone.

is_unclassified: a true/false flag. We keep these tests, so overall failure rates stay complete, and exclude them only when analysing makes or models.

fuel_type_desc: a readable name for each fuel code. ELSE 'Other' catches anything unexpected.

first_use_year: NULLIF(..., 1971) turns the placeholder year into a missing value.

vehicle_age_years: the test year minus the first-use year, and left empty if the year was the 1971 placeholder or came after the test. This is the “age from year only” rule from your dictionary.

The WHERE clause is the same exclusion rule as before. We rebuild from results_dedup (not from the old clean table), so the exclusion and every cleaning rule are applied from one source.

In [106]:
con.sql("""
    SELECT COUNT(*)                                          AS rows_now,
           COUNT(*) FILTER (WHERE vehicle_age_years IS NULL) AS null_age,
           COUNT(*) FILTER (WHERE is_unclassified)           AS unclassified,
           COUNT(*) FILTER (WHERE postcode_area IS NULL)     AS null_postcode
    FROM test_result_clean
""").show()

┌──────────┬──────────┬──────────────┬───────────────┐
│ rows_now │ null_age │ unclassified │ null_postcode │
│  int64   │  int64   │    int64     │     int64     │
├──────────┼──────────┼──────────────┼───────────────┤
│ 42430899 │     3051 │       108292 │          2839 │
└──────────┴──────────┴──────────────┴───────────────┘



How common is 999,999 miles?

In [107]:
con.sql("""
    SELECT test_mileage, COUNT(*) AS n
    FROM test_result_clean
    WHERE test_class_id = '4' AND test_mileage >= 990000
    GROUP BY test_mileage
    ORDER BY n DESC
    LIMIT 15
""").show()

┌──────────────┬───────┐
│ test_mileage │   n   │
│    int64     │ int64 │
├──────────────┼───────┤
│       999999 │   133 │
│       999980 │     3 │
│       993976 │     2 │
│       992243 │     2 │
│       999399 │     2 │
│       990912 │     1 │
│       996943 │     1 │
│       997044 │     1 │
│       997036 │     1 │
│       999950 │     1 │
│       991279 │     1 │
│       999018 │     1 │
│       999457 │     1 │
│       999686 │     1 │
│       993336 │     1 │
└──────────────┴───────┘
  15 rows    2 columns



What other mileage numbers are common?

In [108]:
con.sql("""
    SELECT test_mileage, COUNT(*) AS n
    FROM test_result_clean
    WHERE test_mileage IS NOT NULL
    GROUP BY test_mileage
    ORDER BY n DESC
    LIMIT 15
""").show()

┌──────────────┬───────┐
│ test_mileage │   n   │
│    int64     │ int64 │
├──────────────┼───────┤
│       248601 │   687 │
│            1 │   498 │
│        51295 │   478 │
│        48210 │   468 │
│        50178 │   468 │
│        48585 │   467 │
│        51009 │   467 │
│        55038 │   465 │
│        53408 │   465 │
│        59885 │   464 │
│        45289 │   464 │
│        60012 │   463 │
│        41750 │   463 │
│        45260 │   463 │
│        65125 │   462 │
└──────────────┴───────┘
  15 rows    2 columns



248,601 appears 687 times. That’s well above anything around it. At a mileage that high, individual values should be rare, something may be repeating there.
A mileage of 1 appears 498 times. Almost no vehicle genuinely has one mile on the clock when it’s tested, so this looks like a placeholder.

Who has a mileage of 248,601?

In [109]:
con.sql("""
    SELECT make, model, test_class_id,
           COUNT(*)                   AS tests,
           COUNT(DISTINCT vehicle_id) AS vehicles,
           MIN(test_date)             AS first_test,
           MAX(test_date)             AS last_test
    FROM test_result_clean
    WHERE test_mileage = 248601
    GROUP BY make, model, test_class_id
    ORDER BY tests DESC
    LIMIT 10
""").show()

┌─────────┬─────────┬───────────────┬───────┬──────────┬────────────┬────────────┐
│  make   │  model  │ test_class_id │ tests │ vehicles │ first_test │ last_test  │
│ varchar │ varchar │    varchar    │ int64 │  int64   │    date    │    date    │
├─────────┼─────────┼───────────────┼───────┼──────────┼────────────┼────────────┤
│ CITROEN │ RELAY   │ 7             │   196 │      146 │ 2024-01-02 │ 2024-12-26 │
│ FIAT    │ DUCATO  │ 7             │   157 │      115 │ 2024-01-02 │ 2024-12-27 │
│ PEUGEOT │ BIPPER  │ 4             │    88 │       68 │ 2024-01-03 │ 2024-12-27 │
│ PEUGEOT │ BOXER   │ 7             │    80 │       58 │ 2024-01-04 │ 2024-12-20 │
│ CITROEN │ RELAY   │ 4             │    37 │       30 │ 2024-01-17 │ 2024-11-28 │
│ FIAT    │ DUCATO  │ 4             │    36 │       28 │ 2024-01-06 │ 2024-12-30 │
│ CITROEN │ NEMO    │ 4             │    27 │       21 │ 2024-01-03 │ 2024-12-10 │
│ FIAT    │ FIORINO │ 4             │    21 │       17 │ 2024-01-08 │ 2024-12-19 │
│ PE

Who has a mileage of 1?

In [110]:
con.sql("""
    SELECT make, model,
           COUNT(*)                   AS tests,
           COUNT(DISTINCT vehicle_id) AS vehicles,
           MEDIAN(vehicle_age_years)  AS median_age
    FROM test_result_clean
    WHERE test_mileage = 1
    GROUP BY make, model
    ORDER BY tests DESC
    LIMIT 10
""").show()

┌────────────┬──────────────┬───────┬──────────┬────────────┐
│    make    │    model     │ tests │ vehicles │ median_age │
│  varchar   │   varchar    │ int64 │  int64   │   double   │
├────────────┼──────────────┼───────┼──────────┼────────────┤
│ FORD       │ ESCORT       │     9 │        8 │       44.0 │
│ HONDA      │ CG125        │     7 │        6 │       25.0 │
│ YAMAHA     │ UNCLASSIFIED │     7 │        7 │       30.0 │
│ LAMBRETTA  │ UNCLASSIFIED │     6 │        5 │       46.5 │
│ HONDA      │ C90          │     6 │        5 │       28.5 │
│ YAMAHA     │ YBR 125      │     5 │        5 │       14.0 │
│ LAND ROVER │ DEFENDER     │     5 │        4 │       31.0 │
│ YAMAHA     │ YBR          │     5 │        4 │       10.0 │
│ YAMAHA     │ YBR125       │     5 │        4 │       14.0 │
│ ROVER      │ MINI         │     5 │        5 │       31.0 │
└────────────┴──────────────┴───────┴──────────┴────────────┘
  10 rows                                         5 columns



In [111]:
con.sql("""
    SELECT CASE WHEN test_mileage < 10  THEN '1-9'
                WHEN test_mileage < 100 THEN '10-99'
                ELSE '100-999' END AS band,
           COUNT(*) AS tests
    FROM test_result_clean
    WHERE test_mileage BETWEEN 1 AND 999 AND vehicle_age_years >= 3
    GROUP BY band
    ORDER BY band
""").show()

┌─────────┬───────┐
│  band   │ tests │
│ varchar │ int64 │
├─────────┼───────┤
│ 1-9     │  2015 │
│ 10-99   │  4451 │
│ 100-999 │ 32746 │
└─────────┴───────┘



This restricts to vehicles at least three years old, the age of a first MOT, so a very low mileage is hard to believe. It counts how many fall in each low band.

The CASE sorts each test into a labelled band, and GROUP BY band counts each band. It tells us where a sensible lower cutoff might be.

The mileage rule

I’ll treat a mileage as “no usable reading” (NULL) if it is:

* Under 100 miles, which is a judgement call on my part
* Exactly 248,601 (apparent odometer cap on certain vans)
* Exactly 999,999 (filler, from last time)

In [112]:
con.sql("""
    SELECT COUNT(*) FILTER (WHERE test_mileage < 100)      AS under_100,
           COUNT(*) FILTER (WHERE test_mileage = 248601)   AS spike_248601,
           COUNT(*) FILTER (WHERE test_mileage = 999999)   AS filler_999999
    FROM test_result_clean
""").show()

┌───────────┬──────────────┬───────────────┐
│ under_100 │ spike_248601 │ filler_999999 │
│   int64   │    int64     │     int64     │
├───────────┼──────────────┼───────────────┤
│      7421 │          687 │           137 │
└───────────┴──────────────┴───────────────┘



### Rebuild clean table

In [113]:
con.execute("""
    CREATE OR REPLACE TABLE test_result_clean AS
    WITH typed AS (
        SELECT *,
               TRY_CAST(test_date AS DATE)                          AS test_date_d,
               TRY_CAST(TRY_CAST(test_mileage AS DOUBLE) AS BIGINT) AS mileage_num
        FROM results_dedup
        WHERE test_id NOT IN (SELECT test_id FROM test_conflicts)
    )
    SELECT
        TRY_CAST(test_id AS BIGINT)                          AS test_id,
        TRY_CAST(vehicle_id AS BIGINT)                       AS vehicle_id,
        test_date_d                                          AS test_date,
        test_class_id,
        test_type,
        test_result,
        CASE WHEN test_result IN ('P', 'F', 'PRS')
             THEN test_result IN ('F', 'PRS') END            AS is_initial_fail,
        CASE WHEN mileage_num >= 100
              AND mileage_num NOT IN (248601, 999999)
             THEN mileage_num END                            AS test_mileage,
        NULLIF(postcode_area, 'XX')                          AS postcode_area,
        UPPER(TRIM(make))                                    AS make,
        UPPER(TRIM(model))                                   AS model,
        (UPPER(TRIM(make)) = 'UNCLASSIFIED'
         OR UPPER(TRIM(model)) = 'UNCLASSIFIED')             AS is_unclassified,
        fuel_type,
        CASE fuel_type
            WHEN 'PE' THEN 'Petrol'
            WHEN 'DI' THEN 'Diesel'
            WHEN 'HY' THEN 'Hybrid'
            WHEN 'EL' THEN 'Electric'
            WHEN 'ED' THEN 'Electric Diesel'
            WHEN 'LP' THEN 'LPG'
            WHEN 'GB' THEN 'Gas Bi-Fuel'
            WHEN 'GA' THEN 'Gas'
            WHEN 'GD' THEN 'Gas Diesel'
            WHEN 'FC' THEN 'Fuel Cell'
            WHEN 'CN' THEN 'CNG'
            WHEN 'LN' THEN 'LNG'
            WHEN 'ST' THEN 'Steam'
            ELSE 'Other'
        END                                                  AS fuel_type_desc,
        TRY_CAST(cylinder_capacity AS INTEGER)               AS cylinder_capacity,
        NULLIF(first_use_year, 1971)                         AS first_use_year,
        CASE WHEN first_use_year <> 1971
              AND first_use_year <= YEAR(test_date_d)
             THEN YEAR(test_date_d) - first_use_year
        END                                                  AS vehicle_age_years
    FROM typed
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [114]:
con.sql("""
    SELECT COUNT(*)                                        AS rows_now,
           COUNT(*) FILTER (WHERE test_mileage IS NULL)    AS null_mileage,
           COUNT(*) FILTER (WHERE vehicle_age_years IS NULL) AS null_age
    FROM test_result_clean
""").show()

┌──────────┬──────────────┬──────────┐
│ rows_now │ null_mileage │ null_age │
│  int64   │    int64     │  int64   │
├──────────┼──────────────┼──────────┤
│ 42430899 │       330098 │     3051 │
└──────────┴──────────────┴──────────┘



### Failure item files

In [120]:
con.execute("""
    CREATE OR REPLACE TABLE items_dedup (
        test_id        VARCHAR,
        rfr_id         VARCHAR,
        rfr_type_code  VARCHAR,
        location_id    VARCHAR,
        dangerous_mark VARCHAR,
        filename       VARCHAR
    )
""")

In [116]:
def load_item_month(path):
    con.execute(rf"""
        INSERT INTO items_dedup BY NAME
        SELECT DISTINCT
            test_id,
            rfr_id,
            rfr_type_code,
            mot_test_rfr_location_type_id AS location_id,
            dangerous_mark,
            filename
        FROM read_csv('{path}', all_varchar=true, quote='"', escape='\', filename=true)
        WHERE test_id <> 'test_id'
    """)

Run Cell 1 prior to the the next to guarantee the table starts at 0.

In [121]:
items_folder = "../data/raw/MOT Testing data failure item (2024)"
item_files = sorted(f for f in os.listdir(items_folder) if f.endswith(".csv"))

for f in item_files:
    start = time.time()
    load_item_month(f"{items_folder}/{f}")
    n = con.sql("SELECT COUNT(*) FROM items_dedup").fetchall()[0][0]
    print(f"{f}: table now has {n:,} rows ({time.time() - start:.0f}s)")

test_item_202401.csv: table now has 8,413,821 rows (0s)
test_item_202402.csv: table now has 16,166,546 rows (0s)
test_item_202403.csv: table now has 24,542,746 rows (1s)
test_item_202404.csv: table now has 31,217,828 rows (0s)
test_item_202405.csv: table now has 37,624,650 rows (0s)
test_item_202406.csv: table now has 44,366,318 rows (0s)
test_item_202407.csv: table now has 52,045,388 rows (0s)
test_item_202408.csv: table now has 59,529,433 rows (0s)
test_item_202409.csv: table now has 67,916,572 rows (0s)
test_item_202410.csv: table now has 77,460,590 rows (1s)
test_item_202411.csv: table now has 85,899,697 rows (0s)
test_item_202412.csv: table now has 91,829,954 rows (0s)


In [122]:
con.sql("""
    SELECT COUNT(*)                  AS item_rows,
           COUNT(DISTINCT test_id)   AS tests_with_items,
           COUNT(DISTINCT filename)  AS files
    FROM items_dedup
""").show()

┌───────────┬──────────────────┬───────┐
│ item_rows │ tests_with_items │ files │
│   int64   │      int64       │ int64 │
├───────────┼──────────────────┼───────┤
│  91829954 │         25983976 │    12 │
└───────────┴──────────────────┴───────┘



In [123]:
con.sql("""
    SELECT 'rfr_type_code' AS col, rfr_type_code AS value, COUNT(*) AS n
    FROM items_dedup GROUP BY rfr_type_code
    UNION ALL
    SELECT 'dangerous_mark', dangerous_mark, COUNT(*)
    FROM items_dedup GROUP BY dangerous_mark
    ORDER BY col, n DESC
""").show()

┌────────────────┬─────────┬──────────┐
│      col       │  value  │    n     │
│    varchar     │ varchar │  int64   │
├────────────────┼─────────┼──────────┤
│ dangerous_mark │ NULL    │ 87944884 │
│ dangerous_mark │ D       │  3885070 │
│ rfr_type_code  │ A       │ 60423134 │
│ rfr_type_code  │ F       │ 21455761 │
│ rfr_type_code  │ M       │  6996403 │
│ rfr_type_code  │ P       │  2954656 │
└────────────────┴─────────┴──────────┘



### 'Dangerous' tag deprication

This tag should have been deprecated in 2018 but is in this dataset.

In [124]:
con.sql("""
    SELECT COUNT(*)                                                       AS item_rows,
           COUNT(*) FILTER (WHERE c.test_id IS NOT NULL)                  AS in_clean_tests,
           COUNT(*) FILTER (WHERE x.test_id IS NOT NULL)                  AS in_excluded_tests,
           COUNT(*) FILTER (WHERE c.test_id IS NULL AND x.test_id IS NULL) AS orphans
    FROM items_dedup i
    LEFT JOIN test_result_clean c ON TRY_CAST(i.test_id AS BIGINT) = c.test_id
    LEFT JOIN test_conflicts x    ON i.test_id = x.test_id
""").show()

┌───────────┬────────────────┬───────────────────┬─────────┐
│ item_rows │ in_clean_tests │ in_excluded_tests │ orphans │
│   int64   │     int64      │       int64       │  int64  │
├───────────┼────────────────┼───────────────────┼─────────┤
│  91829954 │       91487420 │            342534 │       0 │
└───────────┴────────────────┴───────────────────┴─────────┘



A LEFT JOIN keeps every item row and attaches the matching test if there is one, otherwise NULL. Here we do it twice: once against our clean tests (c), once against the excluded ones (x).

Each item then falls into exactly one bucket: it belongs to a clean test, to an excluded (conflicting) test, or to neither. The last group, the orphans, would be items pointing at tests that don’t exist in the results.

The three groups must add up to 91,829,954. Expect orphans to be 0 or tiny, and in_excluded_tests to be around 0.4% of the rows.

This joins 92 million rows to 42 million.

In [125]:
con.sql("""
    SELECT rfr_type_code, dangerous_mark, COUNT(*) AS n
    FROM items_dedup
    GROUP BY rfr_type_code, dangerous_mark
    ORDER BY rfr_type_code, n DESC
""").show()

┌───────────────┬────────────────┬──────────┐
│ rfr_type_code │ dangerous_mark │    n     │
│    varchar    │    varchar     │  int64   │
├───────────────┼────────────────┼──────────┤
│ A             │ NULL           │ 60412687 │
│ A             │ D              │    10447 │
│ F             │ NULL           │ 17728139 │
│ F             │ D              │  3727622 │
│ M             │ NULL           │  6996403 │
│ P             │ NULL           │  2807655 │
│ P             │ D              │   147001 │
└───────────────┴────────────────┴──────────┘



In [126]:
con.execute("""
    CREATE OR REPLACE TABLE test_item_clean AS
    SELECT TRY_CAST(test_id AS BIGINT)      AS test_id,
           TRY_CAST(rfr_id AS INTEGER)      AS rfr_id,
           rfr_type_code,
           TRY_CAST(location_id AS INTEGER) AS location_id,
           dangerous_mark
    FROM items_dedup
    WHERE TRY_CAST(test_id AS BIGINT) IN (SELECT test_id FROM test_result_clean)
""")

In [127]:
con.sql("""
    SELECT COUNT(*)                AS item_rows,
           COUNT(DISTINCT test_id) AS tests_with_items
    FROM test_item_clean
""").show()

┌───────────┬──────────────────┐
│ item_rows │ tests_with_items │
│   int64   │      int64       │
├───────────┼──────────────────┤
│  91487420 │         25892109 │
└───────────┴──────────────────┘



item_rows should equal the in_clean_tests number from previous cells. tests_with_items should be a little under the 25.98M from before, since excluded tests drop out.

last values on table were 91487420 and 25892109

At this stage:
* Zero orphans. No item points at a test that doesn’t exist in the results.
* 342,534 items (0.37%) belong to excluded tests. That’s in line with the 0.39% of tests we excluded, so no group of tests is losing a disproportionate share of its items.
* test_item_clean has 91,487,420 items, matching the join check exactly, covering 25,892,109 tests. (About 92,000 tests with items were dropped along with the excluded tests.)

So D is concentrated on fail items, with some on PRS items, never on minors, and on a very small number of advisories. That’s consistent with D flagging dangerous defects, since a dangerous defect causes a fail. But it isn’t conclusive, and the 10,447 advisories with a D don’t fit neatly.

In [129]:
con.close()